# 01. 현재 S&P 500 및 10년 OHLCV 수집
API를 호출하는 유일한 노트북입니다. 최초 수집 또는 명시적인 재수집 때만 실행합니다.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from src.collect_prices import END_DATE, START_DATE, collect_sp500_index, make_df
from src.get_tickers import get_sp500_universe
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f'수집 구간: {START_DATE} ~ {END_DATE} (끝 미포함)')

수집 구간: 2016-01-01 ~ 2026-07-01 (끝 미포함)


In [2]:
sp500_universe = get_sp500_universe()
snp500_tickers = sp500_universe['ticker'].drop_duplicates().tolist()
print(f'수집 대상: {len(snp500_tickers)}종목')
sp500_universe.head()

[캐시] 구성종목 503건 재사용 (C:\workspaces\lab_middle_project\data\raw\cache\sp500_universe.csv)
수집 대상: 503종목


,ticker,company,sector,ticker_original
0,A,Agilent Technologies,Health Care,A
1,AAPL,Apple Inc.,Information Technology,AAPL
2,ABBV,AbbVie,Health Care,ABBV
3,ABNB,Airbnb,Consumer Discretionary,ABNB
4,ABT,Abbott Laboratories,Health Care,ABT


In [3]:
# 오래 걸리는 셀입니다. yfinance -> Yahoo chart -> Tiingo 순으로 복구합니다.
final_df_raw, final_missing_df = make_df(snp500_tickers, start=START_DATE, end=END_DATE, universe=sp500_universe)
print(final_df_raw.shape, final_missing_df.shape)

1차 yfinance:   0%|          | 0/503 [00:00<?, ?it/s]

2차 Yahoo chart:   0%|          | 0/3 [00:00<?, ?it/s]

(1284440, 8) (1, 7)


In [ ]:
# Beta 계산에만 사용할 S&P 500 지수(^GSPC)를 종목 패널과 별도로 수집합니다.
sp500_beta_df = collect_sp500_index(start=START_DATE, end=END_DATE)
print(sp500_beta_df.shape)
display(sp500_beta_df.head())

In [4]:
final_df_raw.to_parquet(RAW_DIR / 'final_df_raw.parquet', index=False)
sp500_beta_df.to_parquet(RAW_DIR / 'sp500_beta_df.parquet', index=False)
final_missing_df.to_csv(RAW_DIR / 'final_missing_df.csv', index=False)
sp500_universe.to_csv(RAW_DIR / 'sp500_universe.csv', index=False)
print(f'raw 저장 완료: {RAW_DIR}')

raw 저장 완료: C:\workspaces\lab_middle_project\data\raw


In [5]:
final_df_raw.head()

,Date,Open,High,Low,Close,Volume,Ticker,source
0,2016-01-04,37.423020,37.590604,36.353055,37.132969,10308400,ABBV,yahoo
1,2016-01-05,37.377907,37.545491,36.623773,36.978279,7179600,ABBV,yahoo
2,2016-01-06,36.333721,37.100747,36.211256,36.984726,8952700,ABBV,yahoo
3,2016-01-07,36.430409,37.281225,36.314388,36.875153,9292600,ABBV,yahoo
4,2016-01-08,37.094310,37.261897,35.760076,35.869652,7985200,ABBV,yahoo


In [6]:
# 실제 데이터 형태
display(final_df_raw.head(10))

# 전체 데이터 규모
print("데이터 크기:", final_df_raw.shape)
print("종목 수:", final_df_raw["Ticker"].nunique())
print("날짜 범위:", final_df_raw["Date"].min(), "~", final_df_raw["Date"].max())

# 수집하지 못한 종목과 사유
display(final_missing_df)

,Date,Open,High,Low,Close,Volume,Ticker,source
0,2016-01-04,37.423020,37.590604,36.353055,37.132969,10308400,ABBV,yahoo
1,2016-01-05,37.377907,37.545491,36.623773,36.978279,7179600,ABBV,yahoo
2,2016-01-06,36.333721,37.100747,36.211256,36.984726,8952700,ABBV,yahoo
3,2016-01-07,36.430409,37.281225,36.314388,36.875153,9292600,ABBV,yahoo
4,2016-01-08,37.094310,37.261897,35.760076,35.869652,7985200,ABBV,yahoo
5,2016-01-11,36.004999,36.082345,34.051990,34.728775,10483300,ABBV,yahoo
6,2016-01-12,35.109052,35.431331,34.528950,35.347538,6799600,ABBV,yahoo
7,2016-01-13,35.529919,35.575512,33.204680,33.334946,10432200,ABBV,yahoo
8,2016-01-14,33.426132,37.014948,33.028821,35.536434,16704900,ABBV,yahoo
9,2016-01-15,34.526883,37.392726,34.526883,37.347134,27008800,ABBV,yahoo


데이터 크기: (1284440, 8)
종목 수: 502
날짜 범위: 2016-01-04 00:00:00 ~ 2026-06-30 00:00:00


,ticker,company,sector,collected,n_rows,fail_stage,fail_reason
0,HONA,Honeywell Aerospace,Industrials,False,0,yahoo+chart,yahoo: rows=11 | chart: None


In [8]:
print(
    "Close 0 이하:",
    (final_df_raw["Close"] <= 0).sum(),
)

Close 0 이하: 0


In [10]:
import pandas as pd 
display(
    pd.DataFrame({
        "결측값": final_df_raw[
            ["Open", "High", "Low", "Volume"]
        ].isna().sum(),
        "0인 값": final_df_raw[
            ["Open", "High", "Low", "Volume"]
        ].eq(0).sum(),
    })
)

,결측값,0인 값
Open,0,0
High,0,0
Low,0,0
Volume,0,1905


# 안정성의 베타지표 계산하기 위해 필요한 benchmark, snp500 지수
- 벤치마크가 없으면 종목 자체의 흔들림인 변동성은 계산할 수 있으나, 시장 대비 민감도인 베타 계산을 위해 필요
- Beta = 1: 시장과 비슷한 정도로 움직임
- -Beta > 1: 시장보다 더 민감하게 움직임
- 0 < Beta < 1: 시장보다 덜 민감하게 움직임
- Beta < 0: 시장과 반대 방향으로 움직이는 경향

In [12]:
import importlib
import src.collect_prices as collect_prices

importlib.reload(collect_prices)

sp500_beta_df = collect_prices.collect_sp500_index(
    start=collect_prices.START_DATE,
    end=collect_prices.END_DATE,
)

In [13]:
sp500_beta_df.to_parquet(
    RAW_DIR / "sp500_beta_df.parquet",
    index=False
)

In [14]:
display(sp500_beta_df.head())
print(sp500_beta_df.shape)

,Date,Close
0,2016-01-04,2012.660034
1,2016-01-05,2016.709961
2,2016-01-06,1990.260010
3,2016-01-07,1943.089966
4,2016-01-08,1922.030029


(2637, 2)
